# 🎮 Reinforcement Learning for Connect-4
## 🧠 An N-Tuple TD(λ) Agent from Scratch

In this workshop we train a **Temporal-Difference (TD) agent** to play Connect-4 to a strong level — without any hand-crafted evaluation function, without minimax search during training, and without labelled game data.

The agent learns purely from **self-play**: it plays games, receives a reward signal (+1 win / 0 draw / −1 loss), and gradually improves its value function.

---

### 🔴 What is Connect-4?

Connect-4 is a two-player, perfect-information, zero-sum board game:

- **Board**: 7 columns × 6 rows, tokens fall to the lowest empty cell in a column
- **Goal**: be the first to place **4 tokens in a row** — horizontally, vertically, or diagonally
- **Players alternate** turns; Yellow (player 1) moves first

---

### 🤔 Why Connect-4 for RL?

| Property | Why it matters |
|---|---|
| Fully deterministic | No hidden information — a clean testbed for value learning |
| Compact state | Entire board fits in 2 × 64-bit integers (bitboard) |
| Non-trivial | Far too large for brute-force; strategy is genuinely required |
| Solved | Ground-truth exists — we can measure how good our agent is |

---

### 📐 The Scale of the Problem

| Fact | Value |
|---|---|
| Total positions | **4,531,985,219,092** (~4.5 trillion) |
| Weak solution | **James Allen** and **Victor Allis** solved it independently in **1988** — just ~two weeks apart! Yellow always wins with perfect play, but a full proof was out of reach at the time |
| Strong solution | **7 years later**, **John Tromp** (Netherlands) completed a full database proof on Sun Microsystems & Silicon Graphics workstations — **40,000 combined CPU-hours** |
| Brute-force at 1,000 pos/s | Would take **~143 years** |
| Brute-force at 1,000,000 pos/s | Still takes **~52 days** |
| Positions per human on Earth | **>500** — more C4 positions than 500× the world's population |

> Our agent cannot search the whole tree. Instead it learns a compact **value function** that scores any position in microseconds.

---

### 🗺️ Notebook Structure

1. **Environment** — Bitboard representation, basic operations, visualisation
2. **State & Rewards** — Board state definition · sparse rewards (+1 / 0 / −1) · deterministic state transitions
3. **Actions** — Legal move generation
4. **Agent & Policy** — N-Tuple network, value function, target network · ε-Greedy action selection
5. **Update** — TD(λ), gradient descent, Adam optimiser


<img src="img/connect4.png" alt="Connect-4 board" />

## 🕹️ Play Connect-4!

Before we dive into the technical details, get a feel for the game first.

The widget below lets you play a round against a **BitBully** agent (minimax, 4-ply search). You play as **Yellow** — click a column to drop your token.

> **Tip:** Try to win — it's harder than it looks!

In [ ]:
%matplotlib ipympl
from bitbully import BitBully
from bitbully.gui_c4 import GuiC4

# Human (Yellow) vs BitBully 4-ply (Red)
agents = {
    "test": BitBully(opening_book=None, tie_break="center", max_depth=4),
}

c4gui = GuiC4(agents=agents, autoplay=True)
display(c4gui.get_widget())

## 🌍 1. Environment

The **environment** is the Connect-4 game itself. It is fully deterministic: given a state and an action, the next state is uniquely determined — no stochasticity (unlike, e.g., Backgammon or card games).

The environment is implemented as `BoardBatch` — a vectorized, PyTorch-native simulator that runs **thousands of games in parallel** on CPU or GPU. What information it stores and what that means in the RL/MDP sense is the subject of **§2 State**. This section focuses on the *mechanics*: how the board is encoded and how tokens are placed, players are swapped, and wins are detected — all using bitwise operations on integers.


### 🔲 1.1 Bitboards

Each board position is encoded as a **single 64-bit integer**. The 7 columns each occupy 9 bits (6 playable rows + 3 sentinel bits):

```
Bit indices in a uint64 (9 bits per column, 7 columns):

         C0    C1    C2    C3    C4    C5    C6
       ┌─────┬─────┬─────┬─────┬─────┬─────┬─────┐
  S2   │  8  │ 17  │ 26  │ 35  │ 44  │ 53  │ 62  │  ← sentinel
  S1   │  7  │ 16  │ 25  │ 34  │ 43  │ 52  │ 61  │  ← sentinel
  S0   │  6  │ 15  │ 24  │ 33  │ 42  │ 51  │ 60  │  ← sentinel
       ├─────┼─────┼─────┼─────┼─────┼─────┼─────┤
  R5   │  5  │ 14  │ 23  │ 32  │ 41  │ 50  │ 59  │  ← top row
  R4   │  4  │ 13  │ 22  │ 31  │ 40  │ 49  │ 58  │
  R3   │  3  │ 12  │ 21  │ 30  │ 39  │ 48  │ 57  │
  R2   │  2  │ 11  │ 20  │ 29  │ 38  │ 47  │ 56  │
  R1   │  1  │ 10  │ 19  │ 28  │ 37  │ 46  │ 55  │
  R0   │  0  │  9  │ 18  │ 27  │ 36  │ 45  │ 54  │  ← bottom row
       └─────┴─────┴─────┴─────┴─────┴─────┴─────┘
```

**Why sentinels?** Legal move generation uses an addition trick. Adding `BB_BOTTOM_ROW` to `all_tokens` propagates carry bits upward through a column. The 3 sentinel bits absorb carry from a full column, preventing it from overflowing into the next column.

$$\text{legal\_moves} = (\texttt{all\_tokens} + \texttt{BB\_BOTTOM\_ROW}) \mathbin{\&} \texttt{BB\_ALL\_LEGAL}$$

### 👁️ 1.2 Visualization Widget

The `BitboardVisualizer` renders the raw bitboard as an interactive Connect-4 board.
Click any column to drop a token and watch the bitboards update in real time.

Available **overlays** (select from the dropdowns):

| Overlay | What it shows |
|---|---|
| `all_tokens` | Every piece on the board — both players combined |
| `active_tokens` | Only the current player's pieces |
| `opponent_tokens` | Derived via XOR — never stored explicitly |
| `legal_moves` | Carry-propagation trick in action |
| `winning_positions` | Cells where a token completes four-in-a-row (threats highlighted) |
| `Bit Index Map` | Which bit in the `int64` corresponds to each cell |

> Use the overlays as you read the sections below — they illustrate every operation interactively.

In [ ]:
%matplotlib ipympl
from techdays26.gui_bitboard import BitboardVisualizer

# Launch an interactive board — click columns to play, select overlay from the dropdown
vis = BitboardVisualizer()
vis.show()

### ⚙️ 1.3 Basic Operations

Three fundamental operations drive the game loop:

#### 🪨 Set stones — `play_columns` / `play_masks`

Placing a token uses the carry-propagation trick introduced in §1.1.
`play_columns(cols)` finds the lowest empty bit in the chosen column and sets it in `all_tokens`.

Select the **`legal_moves`** overlay in the widget above to see the carry trick in action.

In [ ]:
import torch
from techdays26.torch_board import BoardBatch

board = BoardBatch.empty(1)
for col in [3, 3, 3]:
    board.play_columns(torch.tensor([col]))

print("After 3 tokens in column 3:")
print(f"  all_tokens col-3 bits: {(int(board.all_tokens[0]) >> 27) & 0b111111:06b}")
print(f"  moves_left = {int(board.moves_left[0])}")

#### 🔄 Switch players — a single XOR

After every move, `play_columns` / `play_masks` executes:

```python
active_tokens ^= all_tokens
```

Because `all_tokens` now includes the token just placed, XOR-ing with the *previous* `active_tokens` yields the **opponent's** pieces — which become the new `active_tokens` for the next turn. No copy, no extra field.

> Switch between the **`active_tokens`** and **`opponent_tokens`** overlays in the widget to see this in action.

#### 🏆 Detect wins — shift-AND in 4 directions

`has_win()` checks whether the player who just moved has four-in-a-row using the **shift-AND** technique: shift a bitboard by the stride for a given direction, AND with the original, repeat.

| Direction  | Bit stride |
|------------|-----------|
| Horizontal | 9 (column offset) |
| Vertical   | 1 |
| Diagonal ↗ | 10 (col + 1) |
| Diagonal ↘ | 8  (col − 1) |

For any direction with bit-stride *s*:
```python
pairs = y & (y >> s)            # bit i set ↔ tokens at i AND i+s
win   = pairs & (pairs >> 2*s)  # bit i set ↔ four consecutive tokens
```

The entire win check runs in **O(1)** — no loop over cells.

> Select the **`winning_positions`** overlay in the widget to see threats highlighted live.

### ⚡ 1.4 Advantages of the Bitboard Representation

| Property | Benefit |
|---|---|
| **Compact** | Entire game state = 2 × int64 + 1 × int16 — fits in a few registers |
| **Vectorized** | `BoardBatch` stores a whole batch `[B]` of boards as three flat tensors; all operations are elementwise PyTorch ops |
| **GPU-ready** | One `.to("cuda")` call moves the entire batch; every bitwise op runs as a fused GPU kernel with uniform control flow across all boards |
| **O(1) win check** | Four shift-AND operations cover all four directions simultaneously |
| **No copies for player swap** | `active_tokens ^= all_tokens` — one instruction, in-place |
| **Scalable** | Training uses `B = 20 000` parallel games; the bitboard layout makes this practical |

## 🧩 2. State

§1 addressed *how* the game is encoded and simulated — the bitboard layout, token placement, player swap, and win detection.

§2 asks a different question: what is the minimal information that constitutes a **state** in the MDP sense — everything the agent needs to act and be evaluated?

For Connect-4 the answer is exactly three integers:

| Field | Tensor shape | Description |
|---|---|---|
| `all_tokens` | `int64 [B]` | Every piece on the board — both players combined |
| `active_tokens` | `int64 [B]` | The *current player's* pieces only |
| `moves_left` | `int16 [B]` | Remaining moves before the board is full (`42 − tokens placed`) |

These three numbers are **complete**: legal moves, win status, whose turn it is, and reward are all uniquely determined by them. There is no hidden information.

The opponent's pieces are not stored — they are derived on demand: $\text{opponent} = \texttt{active\_tokens} \oplus \texttt{all\_tokens}$

---

### 🔚 Terminal States

| Outcome | Condition |
|---|---|
| Win | Current player completed four-in-a-row |
| Loss | Opponent just won (seen from next player) |
| Draw | Board full, no winner |

We will discuss rewards below!


### 2.1 Inspecting a State


In [ ]:
import torch
from techdays26.torch_board import BoardBatch

# Yellow c3, Red c3, Yellow c4, Red c2 — 4 tokens placed
board = BoardBatch.empty(1, device="cpu")
for col in [3, 3, 4, 2]:
    board.play_columns(torch.tensor([col]))

print("State after cols 3, 3, 4, 2:")
print(f"  all_tokens    = {int(board.all_tokens[0])}")
print(f"  active_tokens = {int(board.active_tokens[0])}  (Yellow — moves next)")
print(f"  moves_left    = {int(board.moves_left[0])}")
print(f"  done()        = {bool(board.done()[0])}")
print(f"  reward()      = {float(board.reward()[0]):.0f}")

### 2.2 💰 Rewards

Connect-4 uses **sparse terminal rewards** — the agent receives a signal only at the end of the game:

| Outcome | Reward |
|---|---|
| Current player completes four-in-a-row | `+1` |
| Opponent just won (next player's perspective) | `−1` |
| Board full, no winner | `0` |

Every intermediate step returns `reward = 0`. This sparsity is what makes the problem hard: the agent must learn to assign credit to early moves that contributed to a win or loss many steps later — the classic **credit assignment problem**.

> **Why no intermediate rewards?** Connect-4 has no natural mid-game progress signal. Any shaping reward (e.g. "reward for three in a row") would encode human domain knowledge — exactly what we want to avoid.


In [ ]:
scenarios = [
    ("Yellow wins (vertical)", [3, 6, 3, 6, 3, 6, 3]),  # Yellow col3×4, Red parks col6
    ("Yellow wins (horizontal)", [0, 6, 1, 6, 2, 6, 3]),  # Yellow cols 0-3 bottom row
    ("Draw (full board)", list(range(7)) * 6),  # fill every column evenly
    ("In-progress", [3, 4]),
]

print(f"  {'Scenario':<28} {'is_done':>8} {'has_win':>8} {'reward':>8}  note")
print("  " + "-" * 72)
for label, seq in scenarios:
    b = BoardBatch.empty(1, device="cpu")
    for col in seq:
        if not bool(b.done()[0]):
            b.play_columns(torch.tensor([col]))
    done = bool(b.done()[0])
    has_win = bool(b.has_win()[0])
    r = float(b.reward()[0])
    # reward() returns from the perspective of the player *about to move* (loser = -1)
    note = {
        (True, True, 1.0): "+1 → player who just moved won",
        (True, True, -1.0): "-1 → player about to move just lost",
        (True, False, 0.0): "draw",
        (False, False, 0.0): "game running",
    }.get((done, has_win, r), "")
    print(f"  {label:<28} {str(done):>8} {str(has_win):>8} {r:>8.0f}  {note}")

### 2.3 🔀 State Transition

The transition function $T(s, a) \rightarrow s'$ is **deterministic**: each (state, action) pair maps to exactly one afterstate, with no randomness involved.

`play_columns(col)` performs the transition in-place in four steps:

| Step | Operation | Effect |
|---|---|---|
| 1 | `landing = (all_tokens + col_bottom) & col_mask` | Carry trick finds the landing bit |
| 2 | `all_tokens \|= landing` | New token added to the board |
| 3 | `active_tokens ^= all_tokens` | Perspective swapped to the opponent |
| 4 | `moves_left -= 1` | Move counter decremented |

After step 3 the board is in the **afterstate** viewed from the *next* player's perspective. The value function $V(s')$ is evaluated on this afterstate — but it scores the position from the perspective of the player who *just moved* (i.e., the agent calling `play_columns`).


In [ ]:
import torch
from techdays26.torch_board import BoardBatch


def snap(b, label):
    print(f"{label}")
    print(
        f"  all_tokens    = {int(b.all_tokens[0]):#018x}  ({bin(int(b.all_tokens[0])).count('1')} bits)"
    )
    print(f"  active_tokens = {int(b.active_tokens[0]):#018x}")
    print(f"  moves_left    = {int(b.moves_left[0])}")


# ── Non-terminal transition ──────────────────────────────────────────────────
board = BoardBatch.empty(1, device="cpu")
for c in [3, 4]:
    board.play_columns(torch.tensor([c]))

snap(board, "State s  (Yellow to move, 2 tokens):")
at_before = int(board.all_tokens[0])
board.play_columns(torch.tensor([3]))
landing = int(board.all_tokens[0]) ^ at_before

print(f"\n── T(s, a=3) ──▶  Afterstate s'\n")
snap(board, "Afterstate s'  (Red to move, 3 tokens):")
print(f"  landing bit   = {landing:#018x}  (bit {landing.bit_length() - 1})")
print(f"  done          = {bool(board.done()[0])}")
print(f"  reward        = {float(board.reward()[0]):.0f}")

# ── Terminal transition ──────────────────────────────────────────────────────
board = BoardBatch.empty(1, device="cpu")
for c in [3, 6, 3, 6, 3, 6]:  # Yellow 3 in a row (col 3); Red parks col 6
    board.play_columns(torch.tensor([c]))

print(f"\n{'=' * 60}")
snap(board, "\nState s  (Yellow to move, one step from winning):")
at_before = int(board.all_tokens[0])
board.play_columns(torch.tensor([3]))  # Yellow completes vertical four
landing = int(board.all_tokens[0]) ^ at_before

print(f"\n── T(s, a=3) ──▶  Terminal afterstate s'\n")
snap(board, "Afterstate s'  (terminal):")
print(f"  landing bit   = {landing:#018x}  (bit {landing.bit_length() - 1})")
print(f"  has_win       = {bool(board.has_win()[0])}")
print(f"  done          = {bool(board.done()[0])}")
print(f"  reward        = {float(board.reward()[0]):.0f}  ← +1 for the winner")

## 🎯 3. Actions

An **action** in Connect-4 is simply dropping a token into one of the 7 columns. But not every column is available, and not every legal move is *sensible*. This section covers the three layers of move generation — from raw legality to strategic pruning.

### 3.1 Legal Moves

A move is **legal** if the column is not full. `generate_legal_moves()` returns a bitboard with one bit set per legal column — the landing square where the token would fall.

This is the carry trick from §1.1:

$$\text{legal\_moves} = (\texttt{all\_tokens} + \texttt{BB\_BOTTOM\_ROW}) \mathbin{\&} \texttt{BB\_ALL\_LEGAL}$$

> Select the **`legal_moves`** overlay in the widget (§1.2) to see this live.

In [ ]:
import torch
from techdays26.torch_board import BoardBatch

board = BoardBatch.empty(1)
for col in [3, 3, 3, 3, 3, 3]:  # fill column 3 completely
    board.play_columns(torch.tensor([col]))

legal = board.generate_legal_moves()
legal_int = int(legal[0])

# Extract which columns have a legal move
open_cols = [c for c in range(7) if (legal_int >> (c * 9)) & 0x3F]
print(f"After filling column 3:  legal columns = {open_cols}")
print(f"Column 3 open? {3 in open_cols}  (full!)")

### 3.2 Non-Losing Moves

`generate_non_losing_moves()` prunes legal moves that would let the opponent win immediately:

1. Compute the opponent's **winning positions** (squares that complete a four-in-a-row)
2. If the opponent has a direct threat, the agent *must* block it — all other moves are pruned
3. If the opponent has **two or more** direct threats, no move can save the game → returns `0`
4. Never place a token directly **below** an opponent's threat (it hands them the win next turn)

This is not search — it's a cheap one-ply safety filter that prevents obvious blunders even during random exploration.

In [ ]:
# Build a position where Red threatens to win in column 4
board = BoardBatch.empty(1)
for col in [0, 4, 1, 4, 2, 4]:  # Red has 3 in col 4 (rows 0-2)
    board.play_columns(torch.tensor([col]))
# Yellow to move — Red threatens col 4, row 3

legal = board.generate_legal_moves()
non_losing = board.generate_non_losing_moves()


def cols_from_mask(mask_int):
    return [c for c in range(7) if (mask_int >> (c * 9)) & 0x3F]


print("Red has 3-in-a-row in column 4 — Yellow must block!")
print(f"  Legal columns:       {cols_from_mask(int(legal[0]))}")
print(f"  Non-losing columns:  {cols_from_mask(int(non_losing[0]))}")

### 3.3 Iterating Over Moves

`iter_move_masks(moves)` yields **one-hot bitboards** — one per legal column — from a move-set bitboard. It extracts the lowest set bit, yields it, and clears it:

```python
mv = moves & -moves   # extract lowest set bit
moves ^= mv           # clear it
```

This is the loop the agent uses in §4 to evaluate each afterstate and pick the best move.

In [ ]:
board = BoardBatch.empty(1)
for col in [3, 4]:
    board.play_columns(torch.tensor([col]))

print("Iterating over non-losing moves:")
moves = board.generate_non_losing_moves()
for mv in board.iter_move_masks(moves):
    if not (mv != 0).any():
        break
    bit = int(mv[0]).bit_length() - 1
    col = bit // 9
    row = bit % 9
    print(f"  move bit {bit:2d} → column {col}, row {row}")

## 🤖 4. Agent & Policy

The agent needs two things: a **value function** that scores any board position, and a **policy** that uses those scores to pick moves. This section covers both.

### 4.1 N-Tuple Network

An N-Tuple Network is a sum of **Look-Up Tables (LUTs)**. Each tuple is a fixed pattern of $N$ board cells. For a given board state:

1. Read the $N$ cells — each is one of 4 states: `empty`, `yellow`, `red`, or `reachable` (empty + legal move landing)
2. Encode the $N$ cell values as a **base-4 index** $T \in [0, 4^N)$
3. Look up weight $W[T]$ in that tuple's LUT
4. **Sum** over all $M$ tuples (plus their horizontal mirrors)
5. Apply $\tanh$ to squash the output to $[-1, +1]$

$$V(s) = \tanh\!\left(\sum_{m=1}^{M} \big[W_m[T_m(s)] + W_m[T_m(\text{mirror}(s))]\big]\right)$$

| Component | Size |
|-----------|------|
| Tuples ($M$) | 200 patterns |
| Cells per tuple ($N$) | 6 |
| LUT entries per tuple ($4^N$) | 4,096 |
| Players | 2 (separate LUTs for Yellow and Red) |
| **Total parameters** | $2 \times 200 \times 4{,}096 = 1{,}638{,}400$ |

The key advantage: **no matrix multiplications** — just integer indexing into arrays. This makes both forward passes and gradient updates extremely fast.

In [ ]:
from techdays26.ntuple_network import NTupleNetwork
from techdays26.ntuples import NTUPLE_BITIDX_LIST_200, format_ntuple, ntuple_summary

info = ntuple_summary(NTUPLE_BITIDX_LIST_200)
print(f"N-Tuple set:  M={info['count']} tuples, N={info['length']} cells each")
print(f"LUT size per tuple: 4^{info['length']} = {info.get('lut_size', '?')}")

net = NTupleNetwork(NTUPLE_BITIDX_LIST_200)
n_params = sum(p.numel() for p in net.parameters())
print(f"Total parameters: {n_params:,}")

print(f"\nExample tuple #0 (cells highlighted with X):")
print(format_ntuple(NTUPLE_BITIDX_LIST_200[0]))

### 4.2 N-Tuple Visualizer

The `NTupleVisualizer` lets you load pretrained weights and inspect the value function interactively:

- **Board panel** — click columns to build positions
- **Pattern overlay** — see which cells each tuple reads (with mirror toggle)
- **LUT heatmap** — the raw weight table for the selected tuple
- **Per-move value bar** — the network's score for each legal move

Load the pretrained model below and explore how the agent evaluates different positions.

In [ ]:
%matplotlib ipympl
from techdays26.gui_ntuple import NTupleVisualizer

vis = NTupleVisualizer(model_path="../td_weights_clean.tdw.zip")
vis.show()

### 4.3 ε-Greedy Afterstate Policy

The agent picks moves using **afterstate evaluation**:

1. For each legal (non-losing) move, **play it on a copy** of the board
2. If the move wins → value = `+1` (terminal reward)
3. Otherwise → evaluate the **afterstate** with the value network
4. **Yellow maximises**, **Red minimises** (zero-sum)
5. With probability $\varepsilon$, pick a **random** legal move instead (exploration)

The `best_afterstate_values()` function does this for an entire batch at once, using reservoir sampling for the random moves — no extra pass needed.

```
   s ─── action a ───▶ s' (afterstate)
                        │
                  V(s') = network score
                        │
         pick a = argmax V(s')  [Yellow]
              or argmin V(s')   [Red]
```

In [ ]:
from techdays26.ntuple_network import NTupleNetwork
from techdays26.torch_board import BoardBatch
from techdays26.training import best_afterstate_values

# Load a pretrained network
net = NTupleNetwork.load("../td_weights_clean.tdw.zip")
net.eval()

# Build a mid-game position
board = BoardBatch.empty(1)
for col in [3, 4, 3, 4, 2]:
    board.play_columns(torch.tensor([col]))

# Evaluate all afterstates — greedy (no exploration)
chosen_mv, chosen_val = best_afterstate_values(board, net)
bit = int(chosen_mv[0]).bit_length() - 1
print(f"Best move: column {bit // 9} (bit {bit})")
print(f"Afterstate value: {float(chosen_val[0]):.4f}")

# Show all move values
print(f"\nAll move evaluations:")
moves = board.generate_non_losing_moves()
for mv in board.iter_move_masks(moves):
    if not (mv != 0).any():
        break
    tmp = BoardBatch(
        all_tokens=board.all_tokens.clone(),
        active_tokens=board.active_tokens.clone(),
        moves_left=board.moves_left.clone(),
    )
    tmp.play_masks(mv)
    v = float(net(tmp)[0])
    bit = int(mv[0]).bit_length() - 1
    col = bit // 9
    print(f"  col {col}: V = {v:+.4f}")

## 📉 5. Update Rule — TD(λ)

The agent improves by playing games against itself and updating its value function after every move. The update rule is **Temporal-Difference learning with eligibility traces** — TD(λ).

### 5.1 TD Error

After the agent plays move $a$ in state $s$, producing afterstate $s'$, we compare two estimates:

$$\delta = \underbrace{r + V_{\text{target}}(s')}_{\text{TD target}} - \underbrace{V(s_{\text{prev}})}_{\text{previous estimate}}$$

- $V(s_{\text{prev}})$ — value of the *previous* afterstate (the position the opponent saw)
- $r$ — reward (`+1` / `−1` at game end, `0` otherwise)
- $V_{\text{target}}(s')$ — bootstrap from a **frozen target network** (Polyak-averaged copy of the online network)

The **afterstate** formulation is key: we evaluate positions *after* our move but *before* the opponent replies. This halves the number of network evaluations compared to state-value TD.

For terminal states, no bootstrapping is needed: the target is simply the reward $r$.

### 5.2 TD(λ) — Eligibility Traces via Truncated Returns

Plain TD (λ=0) only propagates information one step at a time — slow for a 42-move game. TD(λ) blends multi-step returns using a decay factor $\lambda \in [0, 1]$:

$$G_t^{\lambda} = (1 - \lambda) \sum_{n=1}^{\infty} \lambda^{n-1} G_t^{(n)}$$

In practice we use a **truncated forward view** with a ring buffer of the last $k$ afterstate values. When the buffer is full (or a game ends), we fold λ-weighted returns backward:

```
Step:     t    t+1    t+2    t+3   ...   t+k
Value:   V₀     V₁     V₂     V₃   ...   Vₖ
                                          ↑ bootstrap
Target for V₀:  (1-λ)·V₁ + (1-λ)·λ·V₂ + (1-λ)·λ²·V₃ + ... + λᵏ·Vₖ
```

| Parameter | Role |
|-----------|------|
| $\lambda = 0$ | Pure one-step TD — fast but slow credit assignment |
| $\lambda = 1$ | Monte Carlo — unbiased but high variance |
| $\lambda \approx 0.5$ | Good balance for Connect-4 (our default) |
| $k$ (`n_truncate`) | Ring buffer size — how many steps back we propagate |

### 5.3 Gradient Step

Given the λ-return target $G^\lambda$ for each position in the buffer, the loss is **mean squared error**:

$$\mathcal{L} = \frac{1}{|\mathcal{B}|} \sum_{b \in \mathcal{B}} \big(V_\theta(s_b) - G_b^\lambda\big)^2$$

Moves that were chosen randomly (ε-exploration) are **excluded** from the loss — they provide diverse experience but shouldn't drive the gradient, since their afterstates weren't selected by the current policy.

**Optimiser**: Adam with exponential LR decay from `lr_initial` to `lr_final`.

**Target network**: a Polyak-averaged copy updated each step via:

$$\theta_{\text{target}} \leftarrow \tau \cdot \theta + (1 - \tau) \cdot \theta_{\text{target}}$$

This stabilises the bootstrap targets and prevents oscillation.

> The full training loop is implemented in **[`torch_ntuple_net_lambda.ipynb`](../torch_ntuple_net_lambda.ipynb)** §5.